In [8]:
import numpy as np
import pandas as pd
import warnings
import numpy as np

from itertools import combinations, product
from math import comb
from tqdm.auto import tqdm

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef
)

from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [9]:
df = pd.read_csv(r"../../../data/processed/data_vif.csv")
target = "Risk_Label"

n_total = len(df)
n_train = int(n_total * 0.5)
n_valid = int(n_total * 0.3)
n_test = n_total - n_train - n_valid

#Date 컬럼 제거
df.drop("Date", axis=1, inplace=True)

# Risk_Label 매핑 (High Risk=1, Low Risk=0)
df["Risk_Label"] = df["Risk_Label"].map({"Low Risk": 0, "High Risk": 1})

#train/valid/test 분할 5/3/2
train_df = df.iloc[:n_train].reset_index(drop=True)
valid_df = df.iloc[n_train : n_train + n_valid].reset_index(drop=True)
test_df = df.iloc[n_train + n_valid :].reset_index(drop=True)

print(f"total rows: {n_total}, train: {len(train_df)}, valid: {len(valid_df)}, test: {len(test_df)}")

# Split features/target
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_valid = valid_df.drop(columns=[target])
y_valid = valid_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

print("train shape:", X_train.shape, y_train.shape)
print("valid shape:", X_valid.shape, y_valid.shape)
print("test shape:", X_test.shape, y_test.shape)

total rows: 4108, train: 2054, valid: 1232, test: 822
train shape: (2054, 28) (2054,)
valid shape: (1232, 28) (1232,)
test shape: (822, 28) (822,)


In [ ]:
scaler = MinMaxScaler().set_output(transform="pandas")

# train에만 fit하고 valid/test에는 같은 scaler를 적용
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

print('train/valid/test:', len(X_train), len(X_valid), len(X_test))
print('y_train class:', y_train.value_counts().to_dict())
print('y_valid class:', y_valid.value_counts().to_dict())
print('y_test class:', y_test.value_counts().to_dict())

In [ ]:
# =========================
# Backward Selection Param Search Space
# =========================

param_space = {
    "C": [0.1, 0.5, 1.0, 3.0, 5.0, 10.0],
    "class_weight": ["balanced"],
    "solver": ["liblinear"],
    "threshold": [0.1, 0.2, 0.3, 0.35, 0.40, 0.45, 0.50, 0.55]
}


def make_param_list(param_space):
    keys = list(param_space.keys())
    values = list(param_space.values())

    param_list = []

    for comb in product(*values):
        param_list.append(dict(zip(keys, comb)))

    return param_list


param_list = make_param_list(param_space)

총 파라미터 조합 수: 48


[{'C': 0.1,
  'class_weight': 'balanced',
  'solver': 'liblinear',
  'threshold': 0.1},
 {'C': 0.1,
  'class_weight': 'balanced',
  'solver': 'liblinear',
  'threshold': 0.2},
 {'C': 0.1,
  'class_weight': 'balanced',
  'solver': 'liblinear',
  'threshold': 0.3},
 {'C': 0.1,
  'class_weight': 'balanced',
  'solver': 'liblinear',
  'threshold': 0.35},
 {'C': 0.1,
  'class_weight': 'balanced',
  'solver': 'liblinear',
  'threshold': 0.4}]

In [14]:
def get_metrics(y_true, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    gmean = np.sqrt(recall * spec)

    metrics = {
        "gmean": gmean,
        "acc": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision,
        "recall": recall,
        "spec": spec,
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
        "threshold": threshold,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

    return metrics


def backward_selection_lr(
    X_train,
    y_train,
    X_valid,
    y_valid,
    param_list,
    min_delta=0.001,
    random_state=1,
    max_iter=2000,
    verbose=True
):
    """
    Logistic Regression 기반 후진선택법.
    기준 지표는 validation gmean.

    시작:
        전체 변수를 사용한 모델

    반복:
        변수 하나씩 제거해보고,
        validation gmean이 가장 좋은 제거 후보를 선택.

    종료:
        제거 후 gmean 개선폭이 min_delta 이하이면 중단.
    """

    selected_features = list(X_train.columns)

    best_score = -np.inf
    best_model = None
    best_params = None
    best_metrics = None

    step_history = []
    candidate_history = []

    # =========================
    # 0단계: 전체 변수 기준 성능 먼저 계산
    # =========================
    if verbose:
        print("\n[INITIAL FULL MODEL]")
        print(f"초기 변수 수: {len(selected_features)}")

    for params in param_list:
        threshold = params["threshold"]

        model_params = {
            "C": params["C"],
            "class_weight": params["class_weight"],
            "solver": params["solver"]
        }

        model = LogisticRegression(
            penalty="l2",
            max_iter=max_iter,
            random_state=random_state,
            **model_params
        )

        model.fit(X_train[selected_features], y_train)

        y_proba = model.predict_proba(X_valid[selected_features])[:, 1]

        metrics = get_metrics(
            y_true=y_valid,
            y_proba=y_proba,
            threshold=threshold
        )

        if metrics["gmean"] > best_score:
            best_score = metrics["gmean"]
            best_model = model
            best_params = params
            best_metrics = metrics

    if verbose:
        print(
            f"초기 전체 변수 모델 | "
            f"gmean={best_score:.4f} | "
            f"threshold={best_params['threshold']} | "
            f"C={best_params['C']} | "
            f"class_weight={best_params['class_weight']}"
        )

    step = 1

    # =========================
    # Backward Selection
    # =========================
    while len(selected_features) > 1:
        step_best = None

        if verbose:
            print(f"\n[STEP {step}]")
            print(f"현재 변수 수: {len(selected_features)}")
            print("각 변수를 하나씩 제거하면서 평가 중...")

        for feature in selected_features:
            trial_features = [f for f in selected_features if f != feature]

            for params in param_list:
                threshold = params["threshold"]

                model_params = {
                    "C": params["C"],
                    "class_weight": params["class_weight"],
                    "solver": params["solver"]
                }

                model = LogisticRegression(
                    penalty="l2",
                    max_iter=max_iter,
                    random_state=random_state,
                    **model_params
                )

                model.fit(X_train[trial_features], y_train)

                y_proba = model.predict_proba(X_valid[trial_features])[:, 1]

                metrics = get_metrics(
                    y_true=y_valid,
                    y_proba=y_proba,
                    threshold=threshold
                )

                record = {
                    "step": step,
                    "removed_feature": feature,
                    "features": trial_features,
                    "params": params,
                    "metrics": metrics,
                    "model": model
                }

                candidate_history.append({
                    "step": step,
                    "removed_feature": feature,
                    "n_features": len(trial_features),
                    **params,
                    **metrics
                })

                if step_best is None:
                    step_best = record
                else:
                    if metrics["gmean"] > step_best["metrics"]["gmean"]:
                        step_best = record

        improvement = step_best["metrics"]["gmean"] - best_score

        if improvement > min_delta:
            selected_features = step_best["features"]

            best_score = step_best["metrics"]["gmean"]
            best_model = step_best["model"]
            best_params = step_best["params"]
            best_metrics = step_best["metrics"]

            step_history.append({
                "step": step,
                "removed_feature": step_best["removed_feature"],
                "n_features": len(selected_features),
                "improvement": improvement,
                **best_params,
                **best_metrics
            })

            if verbose:
                print(
                    f"제거 변수: {step_best['removed_feature']} | "
                    f"gmean={best_score:.4f} | "
                    f"improvement={improvement:.6f} | "
                    f"threshold={best_params['threshold']} | "
                    f"C={best_params['C']} | "
                    f"class_weight={best_params['class_weight']}"
                )

            step += 1

        else:
            if verbose:
                print(f"개선폭 {improvement:.6f} <= min_delta {min_delta}")
                print("후진선택 종료")

            break

    result = {
        "selected_features": selected_features,
        "best_model": best_model,
        "best_params": best_params,
        "best_metrics": best_metrics,
        "step_history": pd.DataFrame(step_history),
        "candidate_history": pd.DataFrame(candidate_history)
    }

    return result

In [15]:
backward_result = backward_selection_lr(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    param_list=param_list,
    min_delta=0.001,
    random_state=1,
    max_iter=2000,
    verbose=True
)


print("\n" + "=" * 80)
print("최종 선택 변수")
print("=" * 80)
print(backward_result["selected_features"])


print("\n" + "=" * 80)
print("Best Parameters")
print("=" * 80)
print(backward_result["best_params"])


metric_order = [
    "gmean",
    "threshold",
    "acc",
    "f1",
    "precision",
    "recall",
    "spec",
    "balanced_acc",
    "mcc",
    "roc_auc",
    "pr_auc",
    "tn",
    "fp",
    "fn",
    "tp"
]


print("\n" + "=" * 80)
print("Validation Metrics")
print("=" * 80)

metrics_df = pd.DataFrame([backward_result["best_metrics"]])[metric_order]
display(metrics_df)


print("\n" + "=" * 80)
print("Validation Confusion Matrix")
print("row=true, col=pred")
print("[[TN, FP],")
print(" [FN, TP]]")
print("=" * 80)

cm = np.array([
    [backward_result["best_metrics"]["tn"], backward_result["best_metrics"]["fp"]],
    [backward_result["best_metrics"]["fn"], backward_result["best_metrics"]["tp"]]
])

print(cm)


print("\n" + "=" * 80)
print("Step History")
print("=" * 80)

step_cols = [
    "step",
    "removed_feature",
    "n_features",
    "gmean",
    "improvement",
    "threshold",
    "acc",
    "f1",
    "precision",
    "recall",
    "spec",
    "balanced_acc",
    "tn",
    "fp",
    "fn",
    "tp",
    "C",
    "class_weight",
    "solver"
]

if len(backward_result["step_history"]) > 0:
    display(backward_result["step_history"][step_cols])
else:
    print("제거해도 gmean이 개선되는 변수가 없어서 전체 변수를 유지함.")


# =========================
# Test set 평가
# =========================

try:
    best_model = backward_result["best_model"]
    selected_features = backward_result["selected_features"]
    best_threshold = backward_result["best_params"]["threshold"]

    y_test_proba = best_model.predict_proba(X_test[selected_features])[:, 1]

    test_metrics = get_metrics(
        y_true=y_test,
        y_proba=y_test_proba,
        threshold=best_threshold
    )

    print("\n" + "=" * 80)
    print("Test Metrics")
    print("=" * 80)

    test_metrics_df = pd.DataFrame([test_metrics])[metric_order]
    display(test_metrics_df)

    print("\n" + "=" * 80)
    print("Test Confusion Matrix")
    print("row=true, col=pred")
    print("[[TN, FP],")
    print(" [FN, TP]]")
    print("=" * 80)

    test_cm = np.array([
        [test_metrics["tn"], test_metrics["fp"]],
        [test_metrics["fn"], test_metrics["tp"]]
    ])

    print(test_cm)

except NameError:
    pass


[INITIAL FULL MODEL]
초기 변수 수: 28
초기 전체 변수 모델 | gmean=0.2730 | threshold=0.55 | C=0.1 | class_weight=balanced

[STEP 1]
현재 변수 수: 28
각 변수를 하나씩 제거하면서 평가 중...
제거 변수: KOSPI 200 Volume | gmean=0.6528 | improvement=0.379765 | threshold=0.4 | C=5.0 | class_weight=balanced

[STEP 2]
현재 변수 수: 27
각 변수를 하나씩 제거하면서 평가 중...
제거 변수: KOSPI 200_PPO | gmean=0.6627 | improvement=0.009911 | threshold=0.4 | C=0.5 | class_weight=balanced

[STEP 3]
현재 변수 수: 26
각 변수를 하나씩 제거하면서 평가 중...
제거 변수: return(%) | gmean=0.6703 | improvement=0.007548 | threshold=0.4 | C=0.5 | class_weight=balanced

[STEP 4]
현재 변수 수: 25
각 변수를 하나씩 제거하면서 평가 중...
제거 변수: Signal1_Sell | gmean=0.6740 | improvement=0.003768 | threshold=0.4 | C=0.5 | class_weight=balanced

[STEP 5]
현재 변수 수: 24
각 변수를 하나씩 제거하면서 평가 중...
개선폭 -0.000539 <= min_delta 0.001
후진선택 종료

최종 선택 변수
['KODEX 200_Premium', 'KOSDAQ_return(%)', 'NASDAQ_return(%)', 'Brent Crude Oil_return(%)', 'Gold Spot_return(%)', 'USD/KRW_change(%)', 'KOSPI 200 lagged_1_return(%)', 'KOSPI 200 lagge

,gmean,threshold,acc,f1,precision,recall,spec,balanced_acc,mcc,roc_auc,pr_auc,tn,fp,fn,tp
0,0.674025,0.4,0.609578,0.343793,0.220665,0.777778,0.584112,0.680945,0.245249,0.718622,0.334295,625,445,36,126



Validation Confusion Matrix
row=true, col=pred
[[TN, FP],
 [FN, TP]]
[[625 445]
 [ 36 126]]

Step History


,step,removed_feature,n_features,gmean,improvement,threshold,acc,f1,precision,recall,spec,balanced_acc,tn,fp,fn,tp,C,class_weight,solver
0,1,KOSPI 200 Volume,27,0.652798,0.379765,0.4,0.603896,0.325967,0.209964,0.728395,0.585047,0.656721,626,444,44,118,5.0,balanced,liblinear
1,2,KOSPI 200_PPO,26,0.662709,0.009911,0.4,0.605519,0.334247,0.214789,0.753086,0.583178,0.668132,624,446,40,122,0.5,balanced,liblinear
2,3,return(%),25,0.670257,0.007548,0.4,0.610390,0.340659,0.219081,0.765432,0.586916,0.676174,628,442,38,124,0.5,balanced,liblinear
3,4,Signal1_Sell,24,0.674025,0.003768,0.4,0.609578,0.343793,0.220665,0.777778,0.584112,0.680945,625,445,36,126,0.5,balanced,liblinear



Test Metrics


,gmean,threshold,acc,f1,precision,recall,spec,balanced_acc,mcc,roc_auc,pr_auc,tn,fp,fn,tp
0,0.664263,0.4,0.626521,0.300683,0.190202,0.717391,0.615068,0.66623,0.212218,0.722781,0.288714,449,281,26,66



Test Confusion Matrix
row=true, col=pred
[[TN, FP],
 [FN, TP]]
[[449 281]
 [ 26  66]]


In [16]:
selected_features

['KODEX 200_Premium',
 'KOSDAQ_return(%)',
 'NASDAQ_return(%)',
 'Brent Crude Oil_return(%)',
 'Gold Spot_return(%)',
 'USD/KRW_change(%)',
 'KOSPI 200 lagged_1_return(%)',
 'KOSPI 200 lagged_2_return(%)',
 'VKOSPI_Close',
 'VKOSPI_Change(%)',
 'KOSPI 200_RSI14',
 'KOSPI 200_DMI14',
 'KOSPI 200_ADX14',
 'KOSPI 200_BB_width',
 'KOSPI 200_PPO_Hist',
 'KOSPI 200_OG',
 'GJR_VaR_5_t1',
 'Signal1_Buy',
 'Signal2_Buy',
 'Signal2_Sell',
 'Signal3_Buy',
 'Signal3_Sell',
 'Signal4_Buy',
 'Signal4_Sell']